# 03 — Parse XML → Raw Holdings

Parses every cached information table into one tidy DataFrame (`holdings_raw.parquet`), one row per `<infoTable>` entry, with **all** fields: issuer, title, cusip, value, shares, share_type, put_call, investment_discretion, sole/shared/none.

**Key fixes in `src/parser.py`:**
- **Namespace-safe** parsing — SEC info tables use namespaces with varying prefixes (`ns1:`, `n1:`, none). Matching literal tag names returns 0 rows on namespaced files: the other cause of the old "empty holdings" bug. We match on local tag names.
- **Value-units regime** — before the Jan-2023 technical amendment, `value` was in **thousands of dollars**; after, in whole dollars. We normalize both into `value_usd` so history is comparable.
- Malformed rows are **logged and counted**, never silently swallowed.

In [1]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

Project root: /content/13F-Analytics
Ready.


In [2]:
from src.parser import build_raw_holdings

holdings_raw = build_raw_holdings()
holdings_raw.head(10)

,issuer,title,cusip,value,shares,share_type,put_call,investment_discretion,other_manager,sole,shared,none,value_usd,quarter,report_date,accession_number
0,ALLY FINL INC,COM,02005N100,"498,992,850.00","12,719,675.00",SH,None,DFND,4,"12,719,675.00",0.00,0.00,"498,992,850.00",2026Q1,2026-03-31,0001193125-26-226661
1,ALLY FINL INC,COM,02005N100,"109,996,016.00","2,803,875.00",SH,None,DFND,"2,4,11","2,803,875.00",0.00,0.00,"109,996,016.00",2026Q1,2026-03-31,0001193125-26-226661
2,ALLY FINL INC,COM,02005N100,"165,872,286.00","4,228,200.00",SH,None,DFND,"4,5","4,228,200.00",0.00,0.00,"165,872,286.00",2026Q1,2026-03-31,0001193125-26-226661
3,ALLY FINL INC,COM,02005N100,"123,064,510.00","3,137,000.00",SH,None,DFND,"4,8,11","3,137,000.00",0.00,0.00,"123,064,510.00",2026Q1,2026-03-31,0001193125-26-226661
4,ALLY FINL INC,COM,02005N100,"189,726,088.00","4,836,250.00",SH,None,DFND,"4,10","4,836,250.00",0.00,0.00,"189,726,088.00",2026Q1,2026-03-31,0001193125-26-226661
5,ALLY FINL INC,COM,02005N100,"50,018,250.00","1,275,000.00",SH,None,DFND,"4,11","1,275,000.00",0.00,0.00,"50,018,250.00",2026Q1,2026-03-31,0001193125-26-226661
6,ALPHABET INC,CAP STK CL C,02079K107,"1,028,454,775.00","3,585,215.00",SH,None,DFND,"4,11","3,585,215.00",0.00,0.00,"1,028,454,775.00",2026Q1,2026-03-31,0001193125-26-226661
7,ALPHABET INC,CAP STK CL A,02079K305,"4,802,252.00","16,700.00",SH,None,DFND,"2,4,11","16,700.00",0.00,0.00,"4,802,252.00",2026Q1,2026-03-31,0001193125-26-226661
8,ALPHABET INC,CAP STK CL A,02079K305,"1,926,652,000.00","6,700,000.00",SH,None,DFND,"4,5","6,700,000.00",0.00,0.00,"1,926,652,000.00",2026Q1,2026-03-31,0001193125-26-226661
9,ALPHABET INC,CAP STK CL A,02079K305,"1,797,250,000.00","6,250,000.00",SH,None,DFND,"4,10","6,250,000.00",0.00,0.00,"1,797,250,000.00",2026Q1,2026-03-31,0001193125-26-226661


In [3]:
# Per-quarter row counts — Berkshire typically reports ~40–130 rows/quarter
holdings_raw.groupby("quarter").agg(
    rows=("cusip", "size"),
    issuers=("issuer", "nunique"),
    total_value_usd=("value_usd", "sum"),
)

,rows,issuers,total_value_usd
quarter,,,
2024Q2,129,36,"279,969,062,343.00"
2024Q3,121,37,"266,378,900,503.00"
2024Q4,112,35,"267,175,474,249.00"
2025Q1,4,3,"1,106,550,356.00"
2025Q2,114,37,"257,521,776,925.00"
2025Q3,115,37,"267,334,501,955.00"
2025Q4,110,39,"274,160,086,701.00"
2026Q1,90,26,"263,095,703,570.00"


In [4]:
# Field completeness audit
holdings_raw.isna().mean().round(3).sort_values(ascending=False).to_frame("share_missing")

,share_missing
put_call,1.00
issuer,0.00
cusip,0.00
title,0.00
value,0.00
shares,0.00
share_type,0.00
investment_discretion,0.00
other_manager,0.00
sole,0.00
